# Adaptive Dalitz integration for a narrow $\phi(1020)$ in $B^+\to K^-K^+K^+$

This notebook compares four deterministic normalization strategies for a deliberately simple model,

\[A = c_\phi A_\phi + c_{NR},\]

with a narrow $\phi(1020)\to K^-K^+$ and a constant nonresonant term:

1. uniform ordinary Dalitz grid;
2. adaptive ordinary Dalitz grid;
3. uniform Square-Dalitz grid;
4. adaptive Square-Dalitz grid.

Both adaptive algorithms are driven only by local convergence of the bilinears $F_i^*F_j$ (including the SqDP Jacobian where required). They are not given the pole mass or width.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from dalitzplotfitter import (
    AdaptiveDalitzGrid, AdaptiveSquareDalitzGrid, DalitzGrid,
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
    SquareDalitzGrid, enable_x64,
)

enable_x64()

## 1. Minimal $B^+\to K^-K^+K^+$ model

In [ ]:
channel = DecayChannel('B+', ('K-', 'K+', 'K+'))
PHI_MASS = 1.019461
PHI_WIDTH = 0.004249

model = DecayModel(
    channel,
    [
        Resonance(
            'phi1020', (0, 1), RealImag(1.0, 0.0),
            mass=PHI_MASS, width=PHI_WIDTH, spin=1,
            resonance_radius=1.5, parent_radius=5.0,
        ),
        NonResonant(RealImag(0.35, 0.20), name='NR'),
    ],
    normalize_components=False,
)
print('parent mass:', channel.parent_mass)
print('daughter masses:', channel.daughter_masses)
print('components:', [component.name for component in model.amplitude_model.components])

## 2. Full raw normalization matrix

In [ ]:
def raw_matrix(sample):
    cache = model.prepare_cache(sample, normalization_sample=sample, normalize_components=False)
    return np.asarray(cache.normalization_matrix_fixed)

def relative_matrix_error(matrix, reference, floor=1e-8):
    scale = np.max(np.abs(reference))
    mask = np.abs(reference) > floor*scale
    relative = np.zeros_like(np.abs(reference), dtype=float)
    relative[mask] = np.abs(matrix[mask] - reference[mask]) / np.abs(reference[mask])
    return relative, float(np.max(relative[mask]))

def print_matrix(label, matrix):
    print(label)
    for row in matrix:
        print('  ', '  '.join(f'{z.real:+.6e}{z.imag:+.6e}j' for z in row))

## 3. Dense reference

A dense ordinary equal-area Dalitz grid is used as the common reference. This keeps the reference independent of both adaptive algorithms.

In [ ]:
REFERENCE_N = 700
PAIR = (0, 1)
reference_sample = DalitzGrid(
    channel.parent_mass, channel.daughter_masses, resolution=REFERENCE_N
).sample()
M_reference = raw_matrix(reference_sample)
print('reference points:', reference_sample.size)
print_matrix('M reference', M_reference)

## 4. Uniform-grid convergence

In [ ]:
resolutions = [60, 80, 120, 180, 250, 350, 500]
ordinary_rows = []
square_rows = []
for n in resolutions:
    ordinary = DalitzGrid(channel.parent_mass, channel.daughter_masses, resolution=n).sample()
    _, ordinary_error = relative_matrix_error(raw_matrix(ordinary), M_reference)
    ordinary_rows.append((n, ordinary.size, ordinary_error))

    square = SquareDalitzGrid(
        channel.parent_mass, channel.daughter_masses,
        resolution=n, pair=PAIR, quadrature='midpoint',
    ).sample()
    _, square_error = relative_matrix_error(raw_matrix(square), M_reference)
    square_rows.append((n, square.size, square_error))
    print(f'N={n:4d} ordinary={ordinary_error:.3e}  SqDP={square_error:.3e}')

fig, ax = plt.subplots(figsize=(7,4.5))
ax.loglog([r[1] for r in ordinary_rows], [r[2] for r in ordinary_rows], 'o-', label='uniform Dalitz')
ax.loglog([r[1] for r in square_rows], [r[2] for r in square_rows], 's-', label='uniform SqDP')
ax.set(xlabel='normalization points', ylabel='max relative matrix-element error', title='Uniform-grid convergence')
ax.grid(True, which='both', alpha=.3); ax.legend(); plt.show()

## 5. Adaptive ordinary Dalitz grid

`AdaptiveDalitzGrid` refines the same equal-area auxiliary coordinates $(u,v)$ used internally by `DalitzGrid`. The map has constant Jacobian equal to the total physical Dalitz area, so every generated point is physical and no boundary clipping is needed.

In [ ]:
adaptive_dp = AdaptiveDalitzGrid(
    channel.parent_mass, channel.daughter_masses,
    base_resolution=18, min_depth=1, max_depth=6,
    tolerance=0.02, matrix_floor=1e-9, max_cells=200_000,
).build(model)
M_adaptive_dp = raw_matrix(adaptive_dp.sample)
_, adaptive_dp_error = relative_matrix_error(M_adaptive_dp, M_reference)
print('ordinary adaptive leaves:', adaptive_dp.n_leaves)
print('ordinary adaptive points:', adaptive_dp.size)
print('ordinary max depth:', int(adaptive_dp.leaf_depths.max()))
print('ordinary max matrix error:', adaptive_dp_error)

## 6. Adaptive Square-Dalitz grid

In [ ]:
adaptive_sq = AdaptiveSquareDalitzGrid(
    channel.parent_mass, channel.daughter_masses, pair=PAIR,
    base_resolution=18, min_depth=1, max_depth=6,
    tolerance=0.02, matrix_floor=1e-9, max_cells=200_000,
).build(model)
M_adaptive_sq = raw_matrix(adaptive_sq.sample)
_, adaptive_sq_error = relative_matrix_error(M_adaptive_sq, M_reference)
print('SqDP adaptive leaves:', adaptive_sq.n_leaves)
print('SqDP adaptive points:', adaptive_sq.size)
print('SqDP max depth:', int(adaptive_sq.leaf_depths.max()))
print('SqDP max matrix error:', adaptive_sq_error)

## 7. Where do the algorithms refine?

In [ ]:
def draw_leaf_mesh(bounds, xlabel, ylabel, title):
    segments=[]
    for x0,x1,y0,y1 in bounds:
        segments.extend([[(x0,y0),(x1,y0)],[(x1,y0),(x1,y1)],[(x1,y1),(x0,y1)],[(x0,y1),(x0,y0)]])
    fig,ax=plt.subplots(figsize=(6.5,6.5))
    ax.add_collection(LineCollection(segments, linewidths=.22))
    ax.set(xlim=(0,1),ylim=(0,1),xlabel=xlabel,ylabel=ylabel,title=title)
    ax.set_aspect('equal'); plt.show()

draw_leaf_mesh(adaptive_dp.leaf_bounds, r'$u$', r'$v$', 'Adaptive ordinary-Dalitz leaf cells')
draw_leaf_mesh(adaptive_sq.leaf_bounds, r'$m^\prime$', r'$\theta^\prime$', 'Adaptive SqDP leaf cells')

fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,result,xlabel,ylabel,title in [
    (axes[0],adaptive_dp,r'$u$',r'$v$','ordinary Dalitz'),
    (axes[1],adaptive_sq,r'$m^\prime$',r'$\theta^\prime$','Square Dalitz'),
]:
    b=result.leaf_bounds
    x=.5*(b[:,0]+b[:,1]); y=.5*(b[:,2]+b[:,3])
    sc=ax.scatter(x,y,c=result.leaf_depths,s=5)
    fig.colorbar(sc,ax=ax,label='leaf depth')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel=xlabel,ylabel=ylabel,title=title)
plt.show()

## 8. Accuracy versus cost: all four methods

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.loglog([r[1] for r in ordinary_rows],[r[2] for r in ordinary_rows],'o-',label='uniform Dalitz')
ax.loglog([r[1] for r in square_rows],[r[2] for r in square_rows],'s-',label='uniform SqDP')
ax.scatter([adaptive_dp.size],[adaptive_dp_error],marker='*',s=200,label='adaptive Dalitz')
ax.scatter([adaptive_sq.size],[adaptive_sq_error],marker='P',s=120,label='adaptive SqDP')
ax.set(xlabel='normalization points',ylabel='max relative matrix-element error',title=r'$\phi(1020)+NR$: accuracy versus cost')
ax.grid(True,which='both',alpha=.3); ax.legend(); plt.show()

print(f'adaptive Dalitz: {adaptive_dp.size} points, error={adaptive_dp_error:.3e}')
print(f'adaptive SqDP  : {adaptive_sq.size} points, error={adaptive_sq_error:.3e}')

## 9. Individual matrix elements

In [ ]:
names=['phi1020','NR']
for i,ni in enumerate(names):
    for j,nj in enumerate(names):
        ref=M_reference[i,j]
        dp=M_adaptive_dp[i,j]
        sq=M_adaptive_sq[i,j]
        denom=max(abs(ref),1e-15)
        print(f'{ni:8s} x {nj:8s}: DP err={abs(dp-ref)/denom:.3e}, SqDP err={abs(sq-ref)/denom:.3e}')

## Interpretation

The two adaptive methods use the same numerical principle: compare coarse and four-subcell estimates of every relevant $F_i^*F_j$ matrix element and refine only where the matrix has not converged.

The difference is only the coordinate map. `AdaptiveDalitzGrid` works in the equal-area $(u,v)$ coordinates of the ordinary Dalitz grid, whereas `AdaptiveSquareDalitzGrid` works in $(m',\theta')$ and includes the Square-Dalitz Jacobian locally.

For production fits, the preferred normalization grid should be chosen from the measured matrix-element accuracy versus number of points rather than from the coordinate system alone.